# 7 · A node that runs graphs

An agent offers whole graphs to its model as tools. Conductor has no feature for that, and needs none: the agent is a node, and its function runs each tool graph with the calls a host makes, `graph.run(...)`. This notebook builds one. A small scripted model stands in for a real one, so every step prints the same each time.

In [ ]:
import threading
from collections.abc import Callable
from dataclasses import dataclass, field
from typing import Annotated, Any

from conductor import (
    Ask,
    Choice,
    Context,
    ExternalError,
    Graph,
    InvalidCallError,
    NodeError,
    NodeRegistry,
    Param,
    make_graph,
)

## The context

What only the host has goes in the run's context. Here that is the model the agent talks to, a cancel event and a list of what the tools cost. The agent hands the same object to every graph it runs, so all of it is shared.

In [ ]:
@dataclass(frozen=True)
class Session:
    """What the host gives a run: the model, a cancel event and what the tools cost so far."""

    model: Callable[[list[dict[str, Any]]], dict[str, Any]]
    cancel: threading.Event = field(default_factory=threading.Event)
    costs: list[str] = field(default_factory=list)


registry = NodeRegistry(context_type=Session)

## The tools

Each tool is a graph, compiled once, here at the top and not inside the agent's call. The agent offers only a graph that can run and never asks a person: a question asked inside the agent's call would have nobody to answer it, so `approve` is left out.

In [ ]:
@registry.node("count-words", title="Count words", description="Counts the words in a text.")
def count_words(ctx: Context[Session], text: str) -> int:
    ctx.costs.append("count-words")
    return len(text.split())


@registry.node("fetch", title="Fetch", description="Fetches a page; slow, and stops when the run is cancelled.")
def fetch(ctx: Context[Session], url: str) -> str:
    ctx.costs.append("fetch")
    if ctx.cancel.wait(timeout=2):
        raise ExternalError("Stopped before the page came.")
    return f"<html>{url}</html>"


@registry.node("approve", title="Approve", description="Asks a person to approve a text.", asks=True)
def approve(text: str) -> str:
    raise Ask(f"Approve {text!r}?")


graphs = {
    "count_words": Graph.compile(make_graph(count_words), registry),
    "fetch": Graph.compile(make_graph(fetch), registry),
    "approve": Graph.compile(make_graph(approve), registry),
}
TOOLS = {name: graph for name, graph in graphs.items() if graph.is_runnable and not graph.asks}
print(list(TOOLS))

## What the model is shown

`graph.inputs` describes a graph's inputs the way an editor shows a node's, each with its JSON schema. That is the tool's parameters, as a model's tool call takes them.

In [ ]:
def tool_spec(name: str, graph: Graph[Session]) -> dict[str, Any]:
    return {
        "name": name,
        "parameters": {
            "type": "object",
            "properties": {port.name: port.json_schema for port in graph.inputs.values()},
            "required": [port.name for port in graph.inputs.values() if port.required],
        },
    }


tool_spec("count_words", TOOLS["count_words"])

## The agent

The agent runs a tool graph with `context=ctx` and `cancel=ctx.cancel`, and gives it a `timeout` of its own. Then it reads the run as a host does, and every answer goes back to the model:

- arguments the graph refuses raise `InvalidCallError`, whose `problems` say what is wrong;
- a run that ended early has the `Problem` that ended it in `run.cause`;
- a complete run has its values in `run.outputs`.

Its `tools` input names the graphs it may run. It takes only a constant, so a person picks the tools in the editor.

In [ ]:
ToolName = Annotated[str, Param(choices=[Choice("count_words", "Count words"), Choice("fetch", "Fetch")])]


@registry.node("agent", title="Agent", description="Answers a question with graphs as its tools.")
def agent(
    ctx: Context[Session],
    question: str,
    tools: Annotated[list[ToolName], Param(title="Tools", constant_only=True)],
) -> str:
    messages: list[dict[str, Any]] = [
        {"role": "tools", "content": [tool_spec(name, TOOLS[name]) for name in tools]},
        {"role": "user", "content": question},
    ]
    while "tool" in (reply := ctx.model(messages)):
        run_tool = TOOLS[reply["tool"]]
        try:
            run = run_tool.run(reply["arguments"], context=ctx, cancel=ctx.cancel, timeout=30)
        except InvalidCallError as error:
            result: Any = {"problems": [problem.model_dump(mode="json") for problem in error.problems]}
        else:
            if run.cause is None:
                result = run.outputs
            elif run.status == "cancelled":
                raise NodeError("The tool was cancelled.", ctx={"cause": run.cause.model_dump(mode="json")})
            else:
                result = {"problem": run.cause.model_dump(mode="json")}
        messages.append({"role": "tool", "content": result})
    return str(reply["answer"])

## A scripted model

It calls `count_words` with the wrong argument first, reads the problem, calls it again, and answers.

In [ ]:
def scripted(messages: list[dict[str, Any]]) -> dict[str, Any]:
    last = messages[-1]
    print("model reads:", last["content"] if last["role"] != "tools" else "the tools")
    if last["role"] == "user":
        return {"tool": "count_words", "arguments": {"words": last["content"]}}
    if "problems" in last["content"]:
        return {"tool": "count_words", "arguments": {"text": messages[1]["content"]}}
    return {"answer": f"It has {last['content']['result']} words."}


graph = Graph.compile(
    make_graph(agent, constants={"agent.tools": ["count_words", "fetch"]}), registry
)
session = Session(model=scripted)
run = graph.run({"question": "How many words is this?"}, context=session, cancel=session.cancel)

print(run.status, run.outputs)
print("costs:", session.costs)

## What an editor shows

The agent is one node with its own inputs and outputs. The graphs it runs are not drawn inside it, but its `tools` input lists them in its JSON schema.

In [ ]:
{entry.port.name: entry.port.json_schema for entry in graph.describe().nodes["agent"].inputs}

## Cancelling

The host keeps one event in the context and gives the same one to the outer run. Set it, and the inner run ends `cancelled`, the agent raises, and the outer run ends `cancelled` too. Here a timer stands in for a person pressing stop while `fetch` waits for its page.

In [ ]:
def fetching(messages: list[dict[str, Any]]) -> dict[str, Any]:
    return {"tool": "fetch", "arguments": {"url": "https://example.com"}}


session = Session(model=fetching)
threading.Timer(0.2, session.cancel.set).start()
cancelled = graph.run({"question": "What is on the page?"}, context=session, cancel=session.cancel)

assert cancelled.cause is not None
print(cancelled.status, cancelled.cause.type)

## Running again

The inner runs are the agent's own business: they are not in the outer run's events or its saved run. A retry of the agent, or a call made again, runs its tools again from the start.

A run made from this one reuses the agent's call when its inputs are the same, since conductor matches a call by its inputs and its own code, and the tool graphs are not part of that code. After a tool graph changes, forget the agent to call it again.

In [ ]:
session = Session(model=scripted)
again = run.again(context=session)
print("reused, tools called:", session.costs)

forgotten = run.with_forgotten("agent").resume(context=session)
print("forgotten, tools called:", session.costs)